In [90]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns

In [43]:
dose_response = pd.read_csv("../data/raw/GDSC2_fitted_dose_response.csv")
dose_response.head()

,DATASET,NLME_RESULT_ID,NLME_CURVE_ID,CELL_LINE_NAME,SANGER_MODEL_ID,CANCER_TYPE,DRUG_ID,DRUG_NAME,PUTATIVE_TARGET,PATHWAY_NAME,MIN_CONC,MAX_CONC,LN_IC50,AUC,RMSE,Z_SCORE
0,GDSC2,343,15946310,PFSK-1,SIDM01132,Other Solid Cancers,1003,Camptothecin,TOP1,DNA replication,0.0001,0.1,-1.463887,0.930220,0.089052,0.433123
1,GDSC2,343,15946548,A673,SIDM00848,Ewing's Sarcoma,1003,Camptothecin,TOP1,DNA replication,0.0001,0.1,-4.869455,0.614970,0.111351,-1.421100
2,GDSC2,343,15946830,ES5,SIDM00263,Ewing's Sarcoma,1003,Camptothecin,TOP1,DNA replication,0.0001,0.1,-3.360586,0.791072,0.142855,-0.599569
3,GDSC2,343,15947087,ES7,SIDM00269,Ewing's Sarcoma,1003,Camptothecin,TOP1,DNA replication,0.0001,0.1,-5.044940,0.592660,0.135539,-1.516647
4,GDSC2,343,15947369,EW-11,SIDM00203,Ewing's Sarcoma,1003,Camptothecin,TOP1,DNA replication,0.0001,0.1,-3.741991,0.734047,0.128059,-0.807232


In [44]:
drug_id = 1819 
# Chosen Drug ID: Docetaxel

drug_subset = dose_response[dose_response["DRUG_ID"] == drug_id]
drug_subset.shape

(670, 16)

In [45]:
drug_subset['SANGER_MODEL_ID'].duplicated().sum()

np.int64(0)

In [46]:
path = "../data/raw/rnaseq_merged_rsem_tpm.csv"
header_row = pd.read_csv(path, header=None, nrows=1)
sidms_ids = header_row.iloc[0,3:].tolist()

In [47]:
gene_data = pd.read_csv(path, header=None, skiprows=4)
gene_data.columns = ['gene_symbol', 'ensembl_gene_id', 'gene_id'] + sidms_ids
gene_data = gene_data.set_index('gene_symbol')

In [48]:
drug_subset['HAS_EXPRESSION'] = drug_subset['SANGER_MODEL_ID'].isin(sidms_ids)

In [49]:
drug_subset.head()

,DATASET,NLME_RESULT_ID,NLME_CURVE_ID,CELL_LINE_NAME,SANGER_MODEL_ID,CANCER_TYPE,DRUG_ID,DRUG_NAME,PUTATIVE_TARGET,PATHWAY_NAME,MIN_CONC,MAX_CONC,LN_IC50,AUC,RMSE,Z_SCORE,HAS_EXPRESSION
153655,GDSC2,343,15946460,PFSK-1,SIDM01132,Other Solid Cancers,1819,Docetaxel,Microtubule stabiliser,Mitosis,0.003002,3.0,-4.228011,0.267990,0.193469,-0.716106,True
153656,GDSC2,343,15946725,A673,SIDM00848,Ewing's Sarcoma,1819,Docetaxel,Microtubule stabiliser,Mitosis,0.003002,3.0,-5.961688,0.105028,0.100721,-1.287601,True
153657,GDSC2,343,15947012,ES5,SIDM00263,Ewing's Sarcoma,1819,Docetaxel,Microtubule stabiliser,Mitosis,0.003002,3.0,-4.880559,0.195845,0.249086,-0.931214,True
153658,GDSC2,343,15947264,ES7,SIDM00269,Ewing's Sarcoma,1819,Docetaxel,Microtubule stabiliser,Mitosis,0.003002,3.0,-6.322894,0.082177,0.147412,-1.406670,True
153659,GDSC2,343,15947546,EW-11,SIDM00203,Ewing's Sarcoma,1819,Docetaxel,Microtubule stabiliser,Mitosis,0.003002,3.0,-4.026132,0.302620,0.258046,-0.649558,True


In [50]:
drug_subset['HAS_EXPRESSION'].value_counts()

HAS_EXPRESSION
True     651
False     19
Name: count, dtype: int64

In [51]:
drug_subset = drug_subset[drug_subset['HAS_EXPRESSION']]
drug_subset['HAS_EXPRESSION'].value_counts()

HAS_EXPRESSION
True    651
Name: count, dtype: int64

In [52]:
drug_subset.shape

(651, 17)

In [56]:
overlap_ids = drug_subset['SANGER_MODEL_ID']
gene_data_subset = gene_data[overlap_ids]

In [55]:
gene_data_subset.shape

(41145, 651)

In [61]:
# transposing rna data
X = gene_data_subset.T
X.index.name = 'SANGER_MODEL_ID'
X.shape

(651, 41145)

In [62]:
X.head()

gene_symbol,A1BG,A1BG-AS1,A1CF,A2M,A2M-AS1,A2ML1,A2ML1-AS1,A2ML1-AS2,A2MP1,A3GALT2,...,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11AP1,ZYG11B,ZYX,ZYXP1,ZZEF1,ZZZ3
SANGER_MODEL_ID,,,,,,,,,,,,,,,,,,,,,
SIDM01132,5.1375,3.3840,0.0000,0.6135,0.0000,0.0426,0.0000,0.0,0.0000,0.0841,...,0.4751,1.8992,3.7623,0.9928,0.0,2.8258,6.4130,0.0,4.1554,4.0575
SIDM00848,4.8689,3.4476,0.0000,1.9964,0.1243,3.1890,0.0000,0.0,0.0000,0.0841,...,1.6826,2.3306,5.1863,0.0841,0.0,3.0993,5.5242,NaN,3.6064,4.5211
SIDM00263,5.0365,3.5558,0.0144,3.9855,0.2987,5.7968,0.0566,0.0,0.1375,0.1110,...,2.9486,3.3923,6.5963,4.1787,0.0,4.6809,6.3778,0.0,4.5759,5.6946
SIDM00269,4.5497,3.8001,0.0841,3.2157,0.0566,0.5753,0.0000,0.0,0.0841,0.1506,...,2.8298,3.4128,6.7827,2.3190,0.0,4.2141,9.0105,0.0,4.9340,5.4561
SIDM00203,4.2250,3.2095,0.0144,0.5059,0.0976,2.8156,0.0000,0.0,0.0704,0.2141,...,2.4957,3.4764,5.3071,3.0321,0.0,4.4731,6.9529,0.0,4.3590,5.3366


In [63]:
gene_data_subset.head()

SANGER_MODEL_ID,SIDM01132,SIDM00848,SIDM00263,SIDM00269,SIDM00203,SIDM01111,SIDM00909,SIDM00807,SIDM01085,SIDM01160,...,SIDM00231,SIDM00817,SIDM01265,SIDM00217,SIDM00216,SIDM00214,SIDM00194,SIDM00193,SIDM00498,SIDM00049
gene_symbol,,,,,,,,,,,,,,,,,,,,,
A1BG,5.1375,4.8689,5.0365,4.5497,4.2250,4.4925,4.3806,0.2265,0.0976,1.9260,...,0.0426,3.9736,0.0286,0.1763,0.8156,0.7225,0.0000,0.1375,0.1506,0.3219
A1BG-AS1,3.3840,3.4476,3.5558,3.8001,3.2095,2.8115,3.3992,0.1890,0.2265,1.6276,...,0.0566,1.3730,0.0286,0.0841,0.7824,0.6599,0.2016,0.3561,0.2388,0.2265
A1CF,0.0000,0.0000,0.0144,0.0841,0.0144,0.0144,0.0841,0.0000,0.0286,0.0000,...,0.0000,0.0000,0.0144,0.2510,0.2750,0.0000,1.2928,1.8600,0.0704,3.5814
A2M,0.6135,1.9964,3.9855,3.2157,0.5059,0.4751,3.7355,0.1506,0.2265,0.0000,...,0.0426,0.1110,0.1375,3.0721,0.0000,0.0000,0.0144,0.1110,0.0000,0.0000
A2M-AS1,0.0000,0.1243,0.2987,0.0566,0.0976,0.0286,0.1890,0.0426,0.2987,0.2141,...,0.0704,0.0286,0.1375,0.3219,0.4330,0.3896,0.1506,0.3896,0.0000,0.0841


In [70]:
y = drug_subset.set_index('SANGER_MODEL_ID').loc[X.index, 'AUC']

In [91]:
y.head()

SANGER_MODEL_ID
SIDM01132    0.267990
SIDM00848    0.105028
SIDM00263    0.195845
SIDM00269    0.082177
SIDM00203    0.302620
Name: AUC, dtype: float64

In [92]:
X.shape

(651, 36417)

In [93]:
y.shape

(651,)

In [94]:
(X.index == y.index).all()

np.True_

In [102]:
X.isna().sum().sum()

np.int64(0)

In [104]:

sources = pd.read_csv(path, header=None, skiprows=2, nrows=1)
source_map = dict(zip(sidms_ids, sources.iloc[0, 3:]))

# how many of overlapping cell lines came from each source
pd.Series([source_map[i] for i in X.index]).value_counts()

Broad     397
Sanger    254
Name: count, dtype: int64

In [109]:
missing_counts = X.isna().sum()
X = X.drop(columns=missing_counts[missing_counts > 0].index) # removing missing genes
X.isna().sum().sum()   # should be 0

np.int64(0)

In [110]:
print(f"Samples: {X.shape[0]}")
print (f"Features: {X.shape[1]}")
print ("Target: Docetaxel AUC")

Samples: 651
Features: 36417
Target: Docetaxel AUC


In [111]:
X.to_csv("../data/processed/X_docetaxel.csv")
y.to_csv("../data/processed/y_docetaxel.csv")